# Voice of the Customer with Snowflake AI Functions

This notebook demonstrates using Snowflake Cortex AI SQL functions to extract actionable insights from customer interactions — sentiment analysis, topic classification, structured extraction, and personalized outreach.

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
from snowflake.snowpark.context import get_active_session

session = get_active_session()

random.seed(42)
np.random.seed(42)

In [ ]:
DATABASE = 'ML_DEMO'
SCHEMA = 'PUBLIC'
WAREHOUSE = 'ML_DEMO_WH'
NUM_CUSTOMERS = 50

session.use_database(DATABASE)
session.use_schema(SCHEMA)
session.use_warehouse(WAREHOUSE)

print(f"Using: {DATABASE}.{SCHEMA} on warehouse {WAREHOUSE}")

## Generate Customer Profiles

In [ ]:
first_names = ['James', 'Mary', 'Robert', 'Patricia', 'John', 'Jennifer', 'Michael', 'Linda',
    'David', 'Elizabeth', 'William', 'Barbara', 'Richard', 'Susan', 'Joseph', 'Jessica',
    'Thomas', 'Sarah', 'Christopher', 'Karen', 'Charles', 'Lisa', 'Daniel', 'Nancy',
    'Matthew', 'Betty', 'Anthony', 'Margaret', 'Mark', 'Sandra', 'Donald', 'Ashley',
    'Steven', 'Dorothy', 'Andrew', 'Kimberly', 'Paul', 'Emily', 'Joshua', 'Donna',
    'Kenneth', 'Michelle', 'Kevin', 'Carol', 'Brian', 'Amanda', 'George', 'Melissa',
    'Timothy', 'Deborah']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
    'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson',
    'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin', 'Lee', 'Perez', 'Thompson',
    'White', 'Harris', 'Sanchez', 'Clark', 'Ramirez', 'Lewis', 'Robinson', 'Walker',
    'Young', 'Allen', 'King', 'Wright', 'Scott', 'Torres', 'Nguyen', 'Hill', 'Flores',
    'Green', 'Adams', 'Nelson', 'Baker', 'Hall', 'Rivera', 'Campbell', 'Mitchell',
    'Carter', 'Roberts']

age_groups = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
regions = ['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West']
loyalty_tiers = ['bronze', 'silver', 'gold', 'platinum']

recently_negative_ids = set(random.sample(range(1, NUM_CUSTOMERS + 1), k=int(NUM_CUSTOMERS * 0.2)))

customers = []
for cid in range(1, NUM_CUSTOMERS + 1):
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    customers.append({
        'CUSTOMER_ID': cid,
        'CUSTOMER_NAME': f'{fname} {lname}',
        'EMAIL': f'{fname.lower()}.{lname.lower()}{cid}@example.com',
        'AGE_GROUP': np.random.choice(age_groups, p=[0.10, 0.30, 0.27, 0.18, 0.10, 0.05]),
        'REGION': np.random.choice(regions),
        'LOYALTY_TIER': np.random.choice(loyalty_tiers, p=[0.30, 0.35, 0.25, 0.10]),
        'SIGNUP_DATE': datetime(2023, 1, 1) + timedelta(days=random.randint(0, 365)),
        'RECENTLY_NEGATIVE': cid in recently_negative_ids
    })

customers_df = pd.DataFrame(customers)
print(f"Generated {len(customers_df)} customers ({len(recently_negative_ids)} flagged as recently negative)")
customers_df.head()

## Generate Purchase Transactions

In [ ]:
product_catalog = {
    'electronics': ['Wireless Headphones', 'Bluetooth Speaker', 'Tablet Stand', 'USB-C Hub', 'Portable Charger'],
    'clothing': ['Winter Jacket', 'Running Shoes', 'Cotton T-Shirt', 'Denim Jeans', 'Wool Sweater'],
    'home_goods': ['Scented Candle Set', 'Throw Blanket', 'Kitchen Organizer', 'Ceramic Vase', 'Wall Clock'],
    'beauty': ['Moisturizing Cream', 'Vitamin C Serum', 'Hair Styling Kit', 'Perfume Set', 'Sunscreen SPF 50'],
    'sports': ['Yoga Mat', 'Resistance Bands', 'Water Bottle', 'Fitness Tracker Band', 'Jump Rope'],
    'books': ['Bestseller Novel', 'Cookbook Collection', 'Self-Help Guide', 'History Anthology', 'Science Fiction Box Set'],
    'grocery': ['Organic Coffee Beans', 'Protein Bar Variety Pack', 'Olive Oil Premium', 'Trail Mix Bundle', 'Green Tea Sampler']
}

tier_spend = {'bronze': (15, 60), 'silver': (30, 120), 'gold': (50, 200), 'platinum': (80, 350)}

purchases = []
txn_id = 1
for _, c in customers_df.iterrows():
    num_purchases = random.randint(3, 15)
    lo, hi = tier_spend[c['LOYALTY_TIER']]
    for i in range(num_purchases):
        category = random.choice(list(product_catalog.keys()))
        product = random.choice(product_catalog[category])
        purchase_date = c['SIGNUP_DATE'] + timedelta(days=random.randint(1, 700))
        purchases.append({
            'TRANSACTION_ID': txn_id,
            'CUSTOMER_ID': c['CUSTOMER_ID'],
            'PURCHASE_DATE': purchase_date,
            'PRODUCT_CATEGORY': category,
            'PRODUCT_NAME': product,
            'AMOUNT': round(random.uniform(lo, hi), 2),
            'QUANTITY': random.randint(1, 4)
        })
        txn_id += 1

purchases_df = pd.DataFrame(purchases)
print(f"Generated {len(purchases_df)} purchases")
purchases_df.head()

## Generate Call Transcripts

In [ ]:
positive_call_templates = [
    "Agent: Thank you for calling, how can I help?\nCustomer: Hi, I just wanted to say my {product} arrived and I love it! The quality is amazing.\nAgent: That's wonderful to hear! Is there anything else I can help with?\nCustomer: No, that's it. You guys are great. Thank you!",
    "Agent: Welcome to customer support. What can I do for you?\nCustomer: I had a question about my recent order of {product}. I wanted to add something to it.\nAgent: Sure, I can help with that right away.\nCustomer: Perfect, that was so easy. Thanks for the quick help!",
    "Agent: Hi there, thanks for calling. How can I assist you today?\nCustomer: I wanted to check on the return policy for my {product}. Not that I want to return it — it's great — just curious.\nAgent: No problem! Our return window is 30 days. Glad you're enjoying it!\nCustomer: Awesome, thanks so much!",
    "Agent: Good afternoon, how may I help you?\nCustomer: I just received my {product} and wanted to let you know the packaging was perfect and it got here early!\nAgent: That's great feedback, I'll pass that along to our fulfillment team.\nCustomer: Please do, really impressed with the service."
]

negative_call_templates = [
    "Agent: Thank you for calling, how can I help?\nCustomer: I'm really frustrated. My {product} arrived damaged and nobody has responded to my email for a week.\nAgent: I'm sorry to hear that. Let me look into this.\nCustomer: This is unacceptable. I've been a loyal customer and this is how I'm treated? I want a full refund and I'm considering taking my business elsewhere.",
    "Agent: Welcome to support. What's going on?\nCustomer: I've been waiting 3 weeks for my {product} and the tracking hasn't updated in 10 days. This is ridiculous.\nAgent: I apologize for the delay. Let me check the status.\nCustomer: I needed this for a gift and now it's too late. I'm extremely disappointed. Your shipping is terrible.",
    "Agent: Hi, how can I assist?\nCustomer: You charged me twice for my {product} order! I noticed two charges on my credit card. I called two days ago and was told it would be fixed, but nothing happened.\nAgent: I'm very sorry about that. Let me escalate this immediately.\nCustomer: I shouldn't have to call twice for a billing error. This is really poor service.",
    "Agent: Thank you for calling. How can I help today?\nCustomer: I received the wrong item. I ordered {product} and got something completely different. And your website won't let me start a return.\nAgent: That's not the experience we want you to have. I'll get this sorted out.\nCustomer: I hope so, because I've already spent two hours trying to fix this myself. Very frustrating."
]

products_flat = [p for prods in product_catalog.values() for p in prods]

calls = []
call_id = 1
for _, c in customers_df.iterrows():
    num_calls = random.randint(1, 4)
    is_neg = c['RECENTLY_NEGATIVE']
    for i in range(num_calls):
        product = random.choice(products_flat)
        is_last = (i == num_calls - 1)
        if is_neg and is_last:
            template = random.choice(negative_call_templates)
        else:
            template = random.choice(positive_call_templates)
        call_date = c['SIGNUP_DATE'] + timedelta(days=random.randint(30, 600) + i * 60)
        calls.append({
            'CALL_ID': call_id,
            'CUSTOMER_ID': c['CUSTOMER_ID'],
            'CALL_DATE': call_date,
            'TRANSCRIPT': template.format(product=product)
        })
        call_id += 1

calls_df = pd.DataFrame(calls)
print(f"Generated {len(calls_df)} call transcripts")
calls_df.head()

## Generate Chat Interactions

In [ ]:
positive_chat_templates = [
    "Customer: Hey, quick question about my {product} order — when does it ship?\nAgent: It shipped this morning! You should have it by Thursday.\nCustomer: Nice, thanks for the fast reply!",
    "Customer: Just wanted to say the {product} I got is perfect. Exactly what I wanted.\nAgent: So glad to hear that! Thanks for letting us know.\nCustomer: Will definitely be ordering again soon.",
    "Customer: Can I get a price match on the {product}? I saw it cheaper somewhere else.\nAgent: Absolutely, I've applied a 10% adjustment to your order.\nCustomer: Wow, that was easy. Thank you!"
]

negative_chat_templates = [
    "Customer: My {product} order is STILL not here. It's been 2 weeks. What is going on??\nAgent: I apologize, let me check the tracking for you.\nCustomer: This is the third time I've had to follow up. I want to cancel and get a refund.\nAgent: I understand your frustration. Let me escalate this right away.",
    "Customer: I tried to use my coupon code on {product} and it won't work. Your chat bot was useless.\nAgent: Sorry about that. Can you share the code and I'll apply it manually?\nCustomer: Fine. SAVE20. But honestly this whole experience has been awful.\nAgent: I've applied the code. I'm sorry for the trouble.",
    "Customer: The {product} I received is defective. It stopped working after 2 days.\nAgent: I'm sorry to hear that. I can arrange a replacement for you.\nCustomer: I shouldn't have to deal with this. The quality is really disappointing."
]

chats = []
chat_id = 1
for _, c in customers_df.iterrows():
    num_chats = random.randint(1, 3)
    is_neg = c['RECENTLY_NEGATIVE']
    for i in range(num_chats):
        product = random.choice(products_flat)
        is_last = (i == num_chats - 1)
        if is_neg and is_last:
            template = random.choice(negative_chat_templates)
        else:
            template = random.choice(positive_chat_templates)
        chat_date = c['SIGNUP_DATE'] + timedelta(days=random.randint(30, 600) + i * 45)
        chats.append({
            'CHAT_ID': chat_id,
            'CUSTOMER_ID': c['CUSTOMER_ID'],
            'CHAT_DATE': chat_date,
            'CHAT_TEXT': template.format(product=product)
        })
        chat_id += 1

chats_df = pd.DataFrame(chats)
print(f"Generated {len(chats_df)} chat interactions")
chats_df.head()

## Generate Support Tickets

In [ ]:
positive_ticket_templates = [
    {"subject": "Quick question about {product}", "body": "Hi, I recently purchased a {product} and just had a quick question about the warranty coverage. Everything is working great so far! Just want to make sure I'm covered. Thanks!"},
    {"subject": "Feedback on recent order", "body": "I wanted to reach out and say how happy I am with my {product}. The delivery was fast and the item exceeded my expectations. Keep up the great work!"}
]

negative_ticket_templates = [
    {"subject": "URGENT: Defective {product}", "body": "I purchased a {product} last week and it arrived with a major defect — it doesn't turn on at all. I need an immediate replacement or a full refund. I've already tried contacting support twice with no response. This is unacceptable for a product at this price point. Order #{order_id}."},
    {"subject": "Wrong item shipped - Need resolution", "body": "I ordered a {product} but received a completely different item. I need the correct item shipped ASAP. This is the second time I've had a shipping error. I'm starting to lose confidence in your company. Please fix this immediately. Order #{order_id}."},
    {"subject": "Billing issue - Double charged", "body": "I was charged twice for my {product} purchase. The duplicate charge of ${amount} appeared on my statement. I called about this 5 days ago and was told it would be resolved in 24 hours, but nothing has changed. I need this refund processed today. Order #{order_id}."},
    {"subject": "Delayed shipment complaint", "body": "My {product} order from 3 weeks ago still hasn't arrived. Tracking shows it's been stuck in transit for 12 days. I paid for expedited shipping and this is completely unacceptable. I want a full refund for the shipping cost and a clear answer on where my package is. Order #{order_id}."}
]

tickets = []
ticket_id = 1
for _, c in customers_df.iterrows():
    num_tickets = random.randint(0, 3)
    is_neg = c['RECENTLY_NEGATIVE']
    for i in range(num_tickets):
        product = random.choice(products_flat)
        is_last = (i == num_tickets - 1)
        if is_neg and is_last:
            tmpl = random.choice(negative_ticket_templates)
        else:
            tmpl = random.choice(positive_ticket_templates) if random.random() > 0.3 else random.choice(negative_ticket_templates)
        ticket_date = c['SIGNUP_DATE'] + timedelta(days=random.randint(60, 650) + i * 30)
        order_id = random.randint(100000, 999999)
        amount = round(random.uniform(20, 200), 2)
        tickets.append({
            'TICKET_ID': ticket_id,
            'CUSTOMER_ID': c['CUSTOMER_ID'],
            'TICKET_DATE': ticket_date,
            'SUBJECT': tmpl['subject'].format(product=product),
            'TICKET_BODY': tmpl['body'].format(product=product, order_id=order_id, amount=amount)
        })
        ticket_id += 1

tickets_df = pd.DataFrame(tickets)
print(f"Generated {len(tickets_df)} support tickets")
tickets_df.head()

## Save All Data to Snowflake

In [ ]:
tables = {
    'VOC_CUSTOMERS': customers_df,
    'VOC_PURCHASES': purchases_df,
    'VOC_CALL_TRANSCRIPTS': calls_df,
    'VOC_CHAT_INTERACTIONS': chats_df,
    'VOC_SUPPORT_TICKETS': tickets_df
}

for table_name, df in tables.items():
    session.create_dataframe(df).write.mode('overwrite').save_as_table(table_name)
    print(f"Saved {table_name}: {len(df)} rows")

print(f"\nAll tables saved to {DATABASE}.{SCHEMA}")

## Incremental Dynamic Table Pipeline: Sentiment Analysis

From here on, we use **incremental dynamic tables** so that VoC metrics automatically refresh as new interactions arrive, processing only changed data rather than recomputing everything.

The pipeline is decomposed into intermediate DTs for incremental support:

```
Base Tables --> Sentiment Scoring --> Topic Classification --> Topic Agg --------+
                   |                                                             |
                   +---> Sentiment Agg ----------------------------------> Customer 360 --> Outreach Candidates
                   |                                                             |
             Purchases ---> Purchase Agg ---+                                    |
                   |                        +------------------------------------+
                   +---> Last Purchase -----+
```


We use two complementary functions:
- **SNOWFLAKE.CORTEX.SENTIMENT()** returns a numeric score (-1 to 1) for aggregation and trend analysis
- **AI_SENTIMENT()** with categories provides aspect-level sentiment (agent helpfulness, resolution, product quality)

In [ ]:
session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE VOC_INTERACTION_SENTIMENT
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = {WAREHOUSE}
    AS
SELECT CUSTOMER_ID, CALL_DATE AS INTERACTION_DATE, 'call' AS SOURCE,
    TRANSCRIPT AS TEXT_CONTENT,
    SNOWFLAKE.CORTEX.SENTIMENT(TRANSCRIPT)::NUMBER(10,4) AS SENTIMENT_SCORE,
    AI_SENTIMENT(TRANSCRIPT, ['agent_helpfulness', 'resolution', 'product_quality']) AS CATEGORY_SENTIMENT
FROM VOC_CALL_TRANSCRIPTS
UNION ALL
SELECT CUSTOMER_ID, CHAT_DATE, 'chat',
    CHAT_TEXT,
    SNOWFLAKE.CORTEX.SENTIMENT(CHAT_TEXT)::NUMBER(10,4),
    AI_SENTIMENT(CHAT_TEXT, ['agent_helpfulness', 'resolution', 'product_quality'])
FROM VOC_CHAT_INTERACTIONS
UNION ALL
SELECT CUSTOMER_ID, TICKET_DATE, 'ticket',
    TICKET_BODY,
    SNOWFLAKE.CORTEX.SENTIMENT(TICKET_BODY)::NUMBER(10,4),
    AI_SENTIMENT(TICKET_BODY, ['agent_helpfulness', 'resolution', 'product_quality'])
FROM VOC_SUPPORT_TICKETS
""").collect()

print("Created dynamic table VOC_INTERACTION_SENTIMENT (INCREMENTAL, DOWNSTREAM)")
session.table('VOC_INTERACTION_SENTIMENT').limit(10).to_pandas()

## Dynamic Table: Topic Classification

In [ ]:
session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE VOC_INTERACTION_TOPICS
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = {WAREHOUSE}
    AS
SELECT *,
    AI_CLASSIFY(
        TEXT_CONTENT,
        ['shipping_issue', 'product_defect', 'billing_problem', 'return_request',
         'product_inquiry', 'positive_feedback', 'complaint']
    ) AS TOPIC_CLASSIFICATION
FROM VOC_INTERACTION_SENTIMENT
""").collect()

print("Created dynamic table VOC_INTERACTION_TOPICS (INCREMENTAL, DOWNSTREAM)")
session.table('VOC_INTERACTION_TOPICS').select('CUSTOMER_ID', 'SOURCE', 'SENTIMENT_SCORE', 'TOPIC_CLASSIFICATION').limit(10).to_pandas()

## Dynamic Table: Ticket Extraction

In [ ]:
session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE VOC_TICKET_EXTRACTIONS
    TARGET_LAG = '1 hour'
    WAREHOUSE = {WAREHOUSE}
    AS
SELECT TICKET_ID, CUSTOMER_ID, TICKET_DATE, SUBJECT,
    AI_EXTRACT(
        TICKET_BODY,
        {{
            'product_mentioned': 'What product is mentioned?',
            'issue_type': 'What is the main issue or problem?',
            'urgency': 'How urgent is this request (low, medium, high)?',
            'resolution_requested': 'What resolution does the customer want?'
        }}
    ) AS EXTRACTED
FROM VOC_SUPPORT_TICKETS
""").collect()

print("Created dynamic table VOC_TICKET_EXTRACTIONS")
session.table('VOC_TICKET_EXTRACTIONS').limit(10).to_pandas()

## Intermediate DTs: Pre-Aggregated Metrics

To enable incremental refresh on Customer 360, we decompose the aggregations into separate intermediate dynamic tables.

In [ ]:
# Sentiment aggregation
session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE VOC_SENTIMENT_AGG
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = {WAREHOUSE}
    REFRESH_MODE = INCREMENTAL
    AS
SELECT
    CUSTOMER_ID,
    COUNT(*) OVER (PARTITION BY CUSTOMER_ID) AS TOTAL_INTERACTIONS,
    AVG(SENTIMENT_SCORE) OVER (PARTITION BY CUSTOMER_ID) AS AVG_SENTIMENT,
    SENTIMENT_SCORE AS RECENT_SENTIMENT,
    SOURCE AS RECENT_SOURCE,
    INTERACTION_DATE
FROM VOC_INTERACTION_SENTIMENT
QUALIFY ROW_NUMBER() OVER (PARTITION BY CUSTOMER_ID ORDER BY INTERACTION_DATE DESC) = 1
""").collect()
print("Created VOC_SENTIMENT_AGG (INCREMENTAL, DOWNSTREAM)")

# Topic aggregation
session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE VOC_TOPIC_AGG
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = {WAREHOUSE}
    REFRESH_MODE = INCREMENTAL
    AS
SELECT
    CUSTOMER_ID,
    TOPIC_CLASSIFICATION:labels[0]::STRING AS TOP_TOPIC,
    COUNT(*) AS TOPIC_COUNT
FROM VOC_INTERACTION_TOPICS
GROUP BY CUSTOMER_ID, TOPIC_CLASSIFICATION:labels[0]::STRING
QUALIFY ROW_NUMBER() OVER (PARTITION BY CUSTOMER_ID ORDER BY TOPIC_COUNT DESC) = 1
""").collect()
print("Created VOC_TOPIC_AGG (INCREMENTAL, DOWNSTREAM)")

# Purchase aggregation
session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE VOC_PURCHASE_AGG
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = {WAREHOUSE}
    REFRESH_MODE = INCREMENTAL
    AS
SELECT
    CUSTOMER_ID,
    SUM(AMOUNT)::NUMBER(12,2) AS TOTAL_SPEND,
    AVG(AMOUNT)::NUMBER(10,2) AS AVG_ORDER_VALUE,
    COUNT(*) AS TOTAL_PURCHASES,
    MAX(PURCHASE_DATE) AS LAST_PURCHASE_DATE
FROM VOC_PURCHASES
GROUP BY CUSTOMER_ID
""").collect()
print("Created VOC_PURCHASE_AGG (INCREMENTAL, DOWNSTREAM)")

# Last purchase product name 
session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE VOC_LAST_PURCHASE
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = {WAREHOUSE}
    REFRESH_MODE = INCREMENTAL
    AS
SELECT
    CUSTOMER_ID,
    PRODUCT_NAME AS MOST_RECENT_PURCHASE
FROM VOC_PURCHASES
QUALIFY ROW_NUMBER() OVER (PARTITION BY CUSTOMER_ID ORDER BY PURCHASE_DATE DESC) = 1
""").collect()
print("Created VOC_LAST_PURCHASE (INCREMENTAL, DOWNSTREAM)")

## Dynamic Table: Customer 360 with VoC Metrics

Simple LEFT JOINs of pre-aggregated intermediate DTs. No float aggregations in this query block, so it qualifies for **incremental refresh**.

The prior average sentiment (excluding most recent interaction) is derived mathematically:
`AVG_PRIOR = (AVG_ALL * COUNT - RECENT) / (COUNT - 1)`

In [ ]:
session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE VOC_CUSTOMER_360
    TARGET_LAG = '1 hour'
    WAREHOUSE = {WAREHOUSE}
    REFRESH_MODE = INCREMENTAL
    AS
SELECT
    c.CUSTOMER_ID,
    c.CUSTOMER_NAME,
    c.EMAIL,
    c.LOYALTY_TIER,
    c.REGION,
    sa.TOTAL_INTERACTIONS,
    ROUND(sa.AVG_SENTIMENT, 4) AS AVG_SENTIMENT,
    CASE
        WHEN sa.TOTAL_INTERACTIONS > 1
        THEN ROUND((sa.AVG_SENTIMENT * sa.TOTAL_INTERACTIONS - sa.RECENT_SENTIMENT)
             / (sa.TOTAL_INTERACTIONS - 1), 4)
        ELSE sa.AVG_SENTIMENT
    END AS AVG_SENTIMENT_PRIOR,
    ROUND(sa.RECENT_SENTIMENT, 4) AS RECENT_SENTIMENT,
    CASE
        WHEN sa.TOTAL_INTERACTIONS > 1
        THEN ROUND(sa.RECENT_SENTIMENT - (sa.AVG_SENTIMENT * sa.TOTAL_INTERACTIONS - sa.RECENT_SENTIMENT)
             / (sa.TOTAL_INTERACTIONS - 1), 4)
        ELSE 0
    END AS SENTIMENT_TREND,
    sa.RECENT_SOURCE,
    ta.TOP_TOPIC,
    pa.TOTAL_SPEND,
    pa.AVG_ORDER_VALUE,
    pa.TOTAL_PURCHASES,
    pa.LAST_PURCHASE_DATE,
    lp.MOST_RECENT_PURCHASE
FROM VOC_CUSTOMERS c
LEFT JOIN VOC_SENTIMENT_AGG sa ON c.CUSTOMER_ID = sa.CUSTOMER_ID
LEFT JOIN VOC_TOPIC_AGG ta ON c.CUSTOMER_ID = ta.CUSTOMER_ID
LEFT JOIN VOC_PURCHASE_AGG pa ON c.CUSTOMER_ID = pa.CUSTOMER_ID
LEFT JOIN VOC_LAST_PURCHASE lp ON c.CUSTOMER_ID = lp.CUSTOMER_ID
""").collect()

print("Created dynamic table VOC_CUSTOMER_360 (INCREMENTAL)")
c360 = session.table('VOC_CUSTOMER_360').to_pandas()
print(f"Customer 360 built for {len(c360)} customers")
print(f"\nSentiment trend stats:")
print(c360['SENTIMENT_TREND'].describe())
c360.head(10)

## Identify At-Risk Customers

In [ ]:
at_risk = session.sql("""
WITH ranked_negative AS (
    SELECT s.CUSTOMER_ID, s.INTERACTION_DATE, s.SOURCE, s.TEXT_CONTENT,
        ROW_NUMBER() OVER (PARTITION BY s.CUSTOMER_ID ORDER BY s.INTERACTION_DATE DESC) AS rn
    FROM VOC_INTERACTION_SENTIMENT s
    WHERE s.SENTIMENT_SCORE < 0
)
SELECT c.CUSTOMER_ID, c.CUSTOMER_NAME, c.EMAIL, c.LOYALTY_TIER,
    c.AVG_SENTIMENT, c.RECENT_SENTIMENT, c.SENTIMENT_TREND,
    c.TOP_TOPIC, c.MOST_RECENT_PURCHASE, c.LAST_PURCHASE_DATE, c.TOTAL_SPEND,
    n.SOURCE AS NEGATIVE_SOURCE, n.INTERACTION_DATE AS NEGATIVE_DATE, n.TEXT_CONTENT AS NEGATIVE_INTERACTION
FROM VOC_CUSTOMER_360 c
LEFT JOIN ranked_negative n ON c.CUSTOMER_ID = n.CUSTOMER_ID AND n.rn = 1
WHERE c.SENTIMENT_TREND < -0.3 OR c.RECENT_SENTIMENT < -0.3
ORDER BY c.SENTIMENT_TREND ASC
""").to_pandas()

print(f"Found {len(at_risk)} at-risk customers with declining sentiment")
at_risk

## Personalized Outreach: Incremental DT + Stream + Task

Since `SNOWFLAKE.CORTEX.COMPLETE` cannot be used directly in a Dynamic Table definition (SQL UDF with subquery limitation), we use a hybrid pattern:
1. **Incremental DT** (`VOC_OUTREACH_CANDIDATES`) selects at-risk customers
2. **Stream** captures new/changed candidates
3. **Task** runs `SNOWFLAKE.CORTEX.COMPLETE` to draft emails via a MERGE into the target table

In [ ]:
# Incremental DT: identifies outreach candidates
session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE VOC_OUTREACH_CANDIDATES
    TARGET_LAG = '1 hour'
    WAREHOUSE = {WAREHOUSE}
    REFRESH_MODE = INCREMENTAL
    AS
SELECT
    CUSTOMER_ID,
    CUSTOMER_NAME,
    EMAIL,
    MOST_RECENT_PURCHASE,
    LAST_PURCHASE_DATE,
    SENTIMENT_TREND,
    RECENT_SENTIMENT,
    TOP_TOPIC
FROM VOC_CUSTOMER_360
WHERE SENTIMENT_TREND < -0.3 OR RECENT_SENTIMENT < -0.3
""").collect()
print("Created VOC_OUTREACH_CANDIDATES (INCREMENTAL)")

# Target table for AI-generated emails
session.sql("""
CREATE OR REPLACE TABLE VOC_OUTREACH_EMAILS (
    CUSTOMER_ID NUMBER,
    CUSTOMER_NAME STRING,
    EMAIL STRING,
    MOST_RECENT_PURCHASE STRING,
    SENTIMENT_TREND NUMBER(10,4),
    TOP_TOPIC STRING,
    DRAFT_EMAIL STRING
)
""").collect()
print("Created VOC_OUTREACH_EMAILS table")

# Stream on the incremental DT to capture new/changed candidates
session.sql("""
CREATE OR REPLACE STREAM VOC_OUTREACH_STREAM
    ON DYNAMIC TABLE VOC_OUTREACH_CANDIDATES
    SHOW_INITIAL_ROWS = TRUE
""").collect()
print("Created VOC_OUTREACH_STREAM")

# Task: generates emails for new candidates using SNOWFLAKE.CORTEX.COMPLETE
session.sql(f"""
CREATE OR REPLACE TASK VOC_GENERATE_EMAILS
    WAREHOUSE = {WAREHOUSE}
    SCHEDULE = '60 MINUTE'
    WHEN SYSTEM$STREAM_HAS_DATA('VOC_OUTREACH_STREAM')
AS
MERGE INTO VOC_OUTREACH_EMAILS tgt
USING (
    SELECT
        CUSTOMER_ID,
        CUSTOMER_NAME,
        EMAIL,
        MOST_RECENT_PURCHASE,
        SENTIMENT_TREND,
        TOP_TOPIC,
        SNOWFLAKE.CORTEX.COMPLETE(
            'llama3.1-70b',
            'Draft a brief, empathetic customer service email to ' || CUSTOMER_NAME ||
            ' who recently had a negative experience. Their most recent purchase was a ' ||
            COALESCE(MOST_RECENT_PURCHASE, 'recent order') ||
            ' on ' || COALESCE(TO_CHAR(LAST_PURCHASE_DATE, 'MMMM DD, YYYY'), 'a recent date') ||
            '. Their main concern was related to: ' || COALESCE(TOP_TOPIC, 'their recent experience') ||
            '. Write a sincere apology, reference their purchase, and ask how they are doing. ' ||
            'Keep it under 150 words. Sign it from the Customer Experience Team.'
        ) AS DRAFT_EMAIL
    FROM VOC_OUTREACH_STREAM
    WHERE METADATA$ACTION = 'INSERT'
) src
ON tgt.CUSTOMER_ID = src.CUSTOMER_ID
WHEN MATCHED THEN UPDATE SET
    tgt.CUSTOMER_NAME = src.CUSTOMER_NAME,
    tgt.EMAIL = src.EMAIL,
    tgt.MOST_RECENT_PURCHASE = src.MOST_RECENT_PURCHASE,
    tgt.SENTIMENT_TREND = src.SENTIMENT_TREND,
    tgt.TOP_TOPIC = src.TOP_TOPIC,
    tgt.DRAFT_EMAIL = src.DRAFT_EMAIL
WHEN NOT MATCHED THEN INSERT
    (CUSTOMER_ID, CUSTOMER_NAME, EMAIL, MOST_RECENT_PURCHASE, SENTIMENT_TREND, TOP_TOPIC, DRAFT_EMAIL)
VALUES
    (src.CUSTOMER_ID, src.CUSTOMER_NAME, src.EMAIL, src.MOST_RECENT_PURCHASE, src.SENTIMENT_TREND, src.TOP_TOPIC, src.DRAFT_EMAIL)
""").collect()
print("Created VOC_GENERATE_EMAILS task")

session.sql("ALTER TASK VOC_GENERATE_EMAILS RESUME").collect()
print("Task resumed")

# Execute the task immediately to generate initial emails
session.sql("EXECUTE TASK VOC_GENERATE_EMAILS").collect()
print("Task executed manually - generating emails...")

import time
time.sleep(30)
emails = session.table('VOC_OUTREACH_EMAILS').to_pandas()
print(f"\n{len(emails)} personalized outreach emails generated\n")

for _, row in emails.head(3).iterrows():
    print(f"--- Email to {row['CUSTOMER_NAME']} ({row['EMAIL']}) ---")
    print(row['DRAFT_EMAIL'])
    print()

## CDP / CLV Integration Point

These VoC metrics (sentiment trends, interaction topics, issue frequency) can be pushed to your CDP or joined with CLV predictions to create a complete customer profile.

In [ ]:
session.sql("""
SELECT
    c360.CUSTOMER_ID,
    c360.CUSTOMER_NAME,
    c360.LOYALTY_TIER,
    c360.TOTAL_SPEND,
    c360.AVG_SENTIMENT,
    c360.SENTIMENT_TREND,
    c360.TOP_TOPIC,
    c360.TOTAL_INTERACTIONS,
    DATEDIFF('day', c360.LAST_PURCHASE_DATE, CURRENT_DATE()) AS DAYS_SINCE_LAST_PURCHASE,
    -- These columns would come from a CLV pipeline
    NULL AS CLV_SCORE,
    -- These would come from a CDP
    NULL AS CDP_SEGMENT
FROM VOC_CUSTOMER_360 c360
ORDER BY c360.TOTAL_SPEND DESC
LIMIT 20
""").to_pandas()

## Cortex Search Services: Semantic Search over Unstructured Interactions

We create one **Cortex Search Service** per unstructured document type so the agent can perform semantic (meaning-based) retrieval across call transcripts, chat logs, and support tickets independently.

Each service:
- Indexes the text column for vector similarity search
- Exposes `CUSTOMER_ID` and date as filterable attributes
- Refreshes automatically with a 1-hour target lag

In [ ]:
# Cortex Search Service for Call Transcripts
session.sql(f"""
CREATE OR REPLACE CORTEX SEARCH SERVICE VOC_CALL_SEARCH
    ON TRANSCRIPT
    ATTRIBUTES CUSTOMER_ID, CALL_DATE
    WAREHOUSE = {WAREHOUSE}
    TARGET_LAG = '1 hour'
AS (
    SELECT
        CALL_ID,
        CUSTOMER_ID,
        CALL_DATE,
        TRANSCRIPT
    FROM VOC_CALL_TRANSCRIPTS
)
""").collect()
print("Created Cortex Search Service: VOC_CALL_SEARCH")

# Cortex Search Service for Chat Interactions
session.sql(f"""
CREATE OR REPLACE CORTEX SEARCH SERVICE VOC_CHAT_SEARCH
    ON CHAT_TEXT
    ATTRIBUTES CUSTOMER_ID, CHAT_DATE
    WAREHOUSE = {WAREHOUSE}
    TARGET_LAG = '1 hour'
AS (
    SELECT
        CHAT_ID,
        CUSTOMER_ID,
        CHAT_DATE,
        CHAT_TEXT
    FROM VOC_CHAT_INTERACTIONS
)
""").collect()
print("Created Cortex Search Service: VOC_CHAT_SEARCH")

# Cortex Search Service for Support Tickets
session.sql(f"""
CREATE OR REPLACE CORTEX SEARCH SERVICE VOC_TICKET_SEARCH
    ON TICKET_BODY
    ATTRIBUTES CUSTOMER_ID, TICKET_DATE, SUBJECT
    WAREHOUSE = {WAREHOUSE}
    TARGET_LAG = '1 hour'
AS (
    SELECT
        TICKET_ID,
        CUSTOMER_ID,
        TICKET_DATE,
        SUBJECT,
        TICKET_BODY
    FROM VOC_SUPPORT_TICKETS
)
""").collect()
print("Created Cortex Search Service: VOC_TICKET_SEARCH")

# Verify all services were created
session.sql("SHOW CORTEX SEARCH SERVICES LIKE 'VOC_%'").to_pandas()[['name', 'search_column', 'attribute_columns']]

## Semantic View: Natural Language Interface to Customer 360

A **Semantic View** maps business concepts (dimensions, facts, metrics) onto multiple tables joined via relationships. This enables **Cortex Analyst** to translate natural language questions into correct SQL.

We define **three logical tables**:
1. `c360` (Customer 360) — per-customer aggregates: spend, sentiment, loyalty tier
2. `interactions` (Interaction Topics) — per-interaction detail: channel, sentiment score, topic classification
3. `ticket_details` (Ticket Extractions) — per-ticket AI-extracted fields: urgency, issue type, resolution requested

This lets the agent answer both aggregate questions (*"average sentiment by loyalty tier"*) and channel-specific questions (*"average sentiment across chats"*, *"most common topic in support tickets"*).

In [ ]:
session.sql(f"""
CREATE OR REPLACE SEMANTIC VIEW VOC_CUSTOMER_360_SV

  TABLES (
    c360 AS {DATABASE}.{SCHEMA}.VOC_CUSTOMER_360
      PRIMARY KEY (CUSTOMER_ID)
      WITH SYNONYMS ('customer 360', 'customer profile', 'voc summary')
      COMMENT = 'Unified customer profile combining purchase history, sentiment analysis, and interaction topics',

    interactions AS {DATABASE}.{SCHEMA}.VOC_INTERACTION_TOPICS
      WITH SYNONYMS ('interactions', 'customer interactions', 'voc interactions')
      COMMENT = 'Individual customer interactions across calls, chats, and tickets with sentiment and topic classification',

    ticket_details AS {DATABASE}.{SCHEMA}.VOC_TICKET_EXTRACTIONS
      PRIMARY KEY (TICKET_ID)
      WITH SYNONYMS ('ticket extractions', 'ticket details', 'extracted tickets')
      COMMENT = 'AI-extracted structured fields from support tickets including urgency, issue type, and resolution requested'
  )

  RELATIONSHIPS (
    interactions_to_customer AS
      interactions (CUSTOMER_ID) REFERENCES c360,
    tickets_to_customer AS
      ticket_details (CUSTOMER_ID) REFERENCES c360
  )

  FACTS (
    -- Customer 360 facts
    c360.total_interactions AS TOTAL_INTERACTIONS
      WITH SYNONYMS ('interaction count', 'number of interactions')
      COMMENT = 'Total number of customer interactions across calls, chats, and tickets',
    c360.avg_sentiment_score AS AVG_SENTIMENT
      WITH SYNONYMS ('average sentiment', 'overall sentiment')
      COMMENT = 'Average sentiment score across all interactions, range -1 (negative) to 1 (positive)',
    c360.prior_sentiment AS AVG_SENTIMENT_PRIOR
      WITH SYNONYMS ('previous sentiment', 'historical sentiment')
      COMMENT = 'Average sentiment excluding the most recent interaction',
    c360.latest_sentiment AS RECENT_SENTIMENT
      WITH SYNONYMS ('last sentiment', 'current sentiment')
      COMMENT = 'Sentiment score of the most recent interaction',
    c360.sentiment_change AS SENTIMENT_TREND
      WITH SYNONYMS ('sentiment trend', 'sentiment shift', 'sentiment delta')
      COMMENT = 'Difference between recent sentiment and prior average. Negative values indicate declining satisfaction',
    c360.lifetime_spend AS TOTAL_SPEND
      WITH SYNONYMS ('total spend', 'total revenue', 'lifetime value')
      COMMENT = 'Total dollar amount spent by the customer across all purchases',
    c360.average_order AS AVG_ORDER_VALUE
      WITH SYNONYMS ('average order value', 'avg purchase amount')
      COMMENT = 'Average dollar amount per purchase transaction',
    c360.purchase_count AS TOTAL_PURCHASES
      WITH SYNONYMS ('number of purchases', 'order count')
      COMMENT = 'Total number of purchase transactions',

    -- Interaction-level facts
    interactions.interaction_sentiment AS SENTIMENT_SCORE
      WITH SYNONYMS ('sentiment score', 'interaction sentiment')
      COMMENT = 'Sentiment score for an individual interaction, range -1 to 1',
    interactions.interaction_topic AS TOPIC_CLASSIFICATION:labels[0]::STRING
      WITH SYNONYMS ('topic', 'classified topic', 'interaction topic')
      COMMENT = 'AI-classified topic label for the interaction: shipping_issue, product_defect, billing_problem, return_request, product_inquiry, positive_feedback, or complaint',

    -- Ticket extraction facts
    ticket_details.extracted_product AS EXTRACTED:product_mentioned::STRING
      WITH SYNONYMS ('mentioned product', 'ticket product')
      COMMENT = 'Product name extracted from the ticket text',
    ticket_details.extracted_issue AS EXTRACTED:issue_type::STRING
      WITH SYNONYMS ('issue type', 'ticket issue', 'problem type')
      COMMENT = 'Main issue type extracted from the ticket',
    ticket_details.extracted_urgency AS EXTRACTED:urgency::STRING
      WITH SYNONYMS ('urgency', 'priority', 'urgency level')
      COMMENT = 'Urgency level extracted from the ticket: low, medium, or high',
    ticket_details.extracted_resolution AS EXTRACTED:resolution_requested::STRING
      WITH SYNONYMS ('resolution requested', 'desired resolution', 'customer request')
      COMMENT = 'Resolution the customer requested in the ticket'
  )

  DIMENSIONS (
    -- Customer 360 dimensions
    c360.customer_name AS CUSTOMER_NAME
      WITH SYNONYMS ('name', 'client name')
      COMMENT = 'Full name of the customer',
    c360.email_address AS EMAIL
      WITH SYNONYMS ('email', 'contact email')
      COMMENT = 'Customer email address',
    c360.loyalty_level AS LOYALTY_TIER
      WITH SYNONYMS ('loyalty tier', 'membership level', 'loyalty status')
      COMMENT = 'Customer loyalty program tier: bronze, silver, gold, or platinum',
    c360.customer_region AS REGION
      WITH SYNONYMS ('region', 'geographic region', 'location')
      COMMENT = 'Geographic region: Northeast, Southeast, Midwest, Southwest, or West',
    c360.last_interaction_channel AS RECENT_SOURCE
      WITH SYNONYMS ('recent channel', 'last contact method')
      COMMENT = 'Channel of the most recent interaction: call, chat, or ticket',
    c360.primary_topic AS TOP_TOPIC
      WITH SYNONYMS ('top topic', 'main issue', 'primary concern')
      COMMENT = 'Most frequent interaction topic for this customer',
    c360.last_purchase_date AS LAST_PURCHASE_DATE
      WITH SYNONYMS ('last order date', 'most recent purchase date')
      COMMENT = 'Date of the customers most recent purchase',
    c360.last_product AS MOST_RECENT_PURCHASE
      WITH SYNONYMS ('last product', 'most recent product', 'latest purchase')
      COMMENT = 'Name of the product from the most recent purchase',

    -- Interaction-level dimensions
    interactions.interaction_channel AS SOURCE
      WITH SYNONYMS ('channel', 'source', 'interaction type')
      COMMENT = 'Channel of the interaction: call, chat, or ticket',
    interactions.interaction_date AS INTERACTION_DATE
      WITH SYNONYMS ('interaction date', 'contact date')
      COMMENT = 'Date of the individual interaction',

    -- Ticket-level dimensions
    ticket_details.ticket_date AS TICKET_DATE
      WITH SYNONYMS ('ticket date', 'ticket created date')
      COMMENT = 'Date the support ticket was created',
    ticket_details.ticket_subject AS SUBJECT
      WITH SYNONYMS ('subject', 'ticket subject', 'ticket title')
      COMMENT = 'Subject line of the support ticket'
  )

  METRICS (
    -- Customer-level metrics
    c360.customer_count AS COUNT(CUSTOMER_ID)
      WITH SYNONYMS ('number of customers', 'total customers', 'headcount')
      COMMENT = 'Count of unique customers',
    c360.total_revenue AS SUM(TOTAL_SPEND)
      WITH SYNONYMS ('total revenue', 'gross revenue', 'total sales')
      COMMENT = 'Sum of total spend across all customers',
    c360.avg_customer_sentiment AS AVG(AVG_SENTIMENT)
      WITH SYNONYMS ('average customer sentiment', 'mean sentiment')
      COMMENT = 'Average sentiment score across all customers',
    c360.avg_spend_per_customer AS AVG(TOTAL_SPEND)
      WITH SYNONYMS ('average spend per customer', 'avg lifetime value')
      COMMENT = 'Average total spend per customer',
    c360.total_interaction_count AS SUM(TOTAL_INTERACTIONS)
      WITH SYNONYMS ('total interactions', 'all interactions')
      COMMENT = 'Sum of all interactions across all customers',

    -- Interaction-level metrics
    interactions.interaction_count AS COUNT(*)
      WITH SYNONYMS ('number of interactions', 'interaction volume')
      COMMENT = 'Count of individual interactions (filterable by channel via SOURCE dimension)',
    interactions.avg_interaction_sentiment AS AVG(interactions.interaction_sentiment)
      WITH SYNONYMS ('average interaction sentiment', 'mean interaction sentiment')
      COMMENT = 'Average sentiment score across individual interactions (filterable by channel)',

    -- Ticket-level metrics
    ticket_details.ticket_count AS COUNT(*)
      WITH SYNONYMS ('number of tickets', 'ticket volume', 'support ticket count')
      COMMENT = 'Count of support tickets'
  )

  COMMENT = 'Voice of the Customer semantic view covering customer profiles, per-interaction sentiment and topics, and AI-extracted ticket details. Supports both aggregate and channel-level analytics.'

  AI_SQL_GENERATION 'When the user asks about sentiment or topics for a specific channel (calls, chats, or tickets), use the interactions table filtered by SOURCE. When the user asks about customer-level aggregates (total spend, loyalty tier breakdowns), use the c360 table. For ticket-specific extractions (urgency, issue type), use ticket_details.'
""").collect()

print("Created Semantic View: VOC_CUSTOMER_360_SV")
print("  - 3 logical tables: c360, interactions, ticket_details")
print("  - 2 relationships: interactions -> c360, ticket_details -> c360")
sv_desc = session.sql("DESCRIBE SEMANTIC VIEW VOC_CUSTOMER_360_SV").to_pandas()
print(f"  - {len(sv_desc[sv_desc['object_kind'] == 'FACT'])} facts, "
      f"{len(sv_desc[sv_desc['object_kind'] == 'DIMENSION'])} dimensions, "
      f"{len(sv_desc[sv_desc['object_kind'] == 'METRIC'])} metrics")
sv_desc.head(30)

## Cortex Agent: Unified VoC Assistant

The **Cortex Agent** orchestrates across all VoC data sources in a single conversational interface:

- **Analyst tool** (Cortex Analyst + Semantic View) handles structured questions: *"Which platinum customers have declining sentiment?"*
- **Search tools** (3 Cortex Search Services) handle unstructured retrieval: *"Find call transcripts where customers complained about shipping"*

The agent's orchestration instructions route questions to the right tool automatically.

In [ ]:
session.sql(f"""
CREATE OR REPLACE AGENT VOC_AGENT
  COMMENT = 'Voice of the Customer assistant combining structured analytics with unstructured search'
  FROM SPECIFICATION
  $$
models:
  orchestration: claude-4-sonnet

orchestration:
  budget:
    seconds: 60
    tokens: 16000

instructions:
  system: |
    You are a Voice of the Customer analyst for a retail company.
    You help teams understand customer sentiment, identify at-risk customers,
    analyze purchase patterns, and retrieve specific customer interactions.
  orchestration: |
    Use Analyst for structured questions about metrics, counts, averages, rankings,
    trends, and comparisons (e.g. sentiment scores, spend, customer counts, loyalty tiers).
    Use CallSearch to find specific call transcript content.
    Use ChatSearch to find specific chat interaction content.
    Use TicketSearch to find specific support ticket content.
    When asked about "interactions" or "complaints" in general, search across all three
    sources. When asked about a specific channel, use only that search service.
  response: |
    Be concise and data-driven. Include specific numbers, customer names, and sentiment
    scores when available. When returning search results, quote relevant excerpts.
  sample_questions:
    - question: "Which customers have the most negative sentiment trend?"
      answer: "I'll query the Customer 360 data to find customers with the steepest sentiment decline."
    - question: "Find call transcripts about damaged products"
      answer: "I'll search the call transcript index for mentions of product damage."
    - question: "What is the average sentiment by loyalty tier?"
      answer: "I'll analyze average sentiment scores grouped by loyalty tier."
    - question: "Show me support tickets about billing issues"
      answer: "I'll search the ticket index for billing-related complaints."

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "Analyst"
      description: "Queries structured customer 360 data including sentiment scores, purchase history, loyalty tiers, and interaction metrics. Use for any question about numbers, rankings, or comparisons."
  - tool_spec:
      type: "cortex_search"
      name: "CallSearch"
      description: "Semantic search over customer call transcripts. Use to find calls about specific topics, products, or issues."
  - tool_spec:
      type: "cortex_search"
      name: "ChatSearch"
      description: "Semantic search over customer chat interactions. Use to find chats about specific topics, products, or issues."
  - tool_spec:
      type: "cortex_search"
      name: "TicketSearch"
      description: "Semantic search over customer support tickets. Use to find tickets about specific topics, products, or issues."

tool_resources:
  Analyst:
    semantic_view: "{DATABASE}.{SCHEMA}.VOC_CUSTOMER_360_SV"
    execution_environment:
      type: "warehouse"
      warehouse: "{WAREHOUSE}"
  CallSearch:
    name: "{DATABASE}.{SCHEMA}.VOC_CALL_SEARCH"
    max_results: "5"
  ChatSearch:
    name: "{DATABASE}.{SCHEMA}.VOC_CHAT_SEARCH"
    max_results: "5"
  TicketSearch:
    name: "{DATABASE}.{SCHEMA}.VOC_TICKET_SEARCH"
    max_results: "5"
  $$
""").collect()

print("Created Agent: VOC_AGENT")
print("\nAgent is ready. You can interact with it via Snowflake Intelligence")
print("or test it programmatically with SNOWFLAKE.CORTEX.DATA_AGENT_RUN().")

# Test the agent with a sample question
print("\n--- Testing Agent ---")
result = session.sql(f"""
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    '{DATABASE}.{SCHEMA}.VOC_AGENT',
    'Which loyalty tier has the worst average sentiment?'
) AS RESPONSE
""").collect()
import json
response = json.loads(result[0]['RESPONSE'])
print(response.get('message', response))

# Cleanup

In [ ]:
# -- Uncomment and run to drop all VoC objects --

# -- Agent, Semantic View, and Search Services --
# session.sql("DROP AGENT IF EXISTS VOC_AGENT").collect()
# session.sql("DROP SEMANTIC VIEW IF EXISTS VOC_CUSTOMER_360_SV").collect()
# session.sql("DROP CORTEX SEARCH SERVICE IF EXISTS VOC_CALL_SEARCH").collect()
# session.sql("DROP CORTEX SEARCH SERVICE IF EXISTS VOC_CHAT_SEARCH").collect()
# session.sql("DROP CORTEX SEARCH SERVICE IF EXISTS VOC_TICKET_SEARCH").collect()

# -- Task, Stream, and Outreach --
# session.sql("ALTER TASK IF EXISTS VOC_GENERATE_EMAILS SUSPEND").collect()
# session.sql("DROP TASK IF EXISTS VOC_GENERATE_EMAILS").collect()
# session.sql("DROP STREAM IF EXISTS VOC_OUTREACH_STREAM").collect()
# session.sql("DROP TABLE IF EXISTS VOC_OUTREACH_EMAILS").collect()

# -- Dynamic Tables (reverse dependency order) --
# session.sql("DROP DYNAMIC TABLE IF EXISTS VOC_OUTREACH_CANDIDATES").collect()
# session.sql("DROP DYNAMIC TABLE IF EXISTS VOC_CUSTOMER_360").collect()
# session.sql("DROP DYNAMIC TABLE IF EXISTS VOC_SENTIMENT_AGG").collect()
# session.sql("DROP DYNAMIC TABLE IF EXISTS VOC_TOPIC_AGG").collect()
# session.sql("DROP DYNAMIC TABLE IF EXISTS VOC_PURCHASE_AGG").collect()
# session.sql("DROP DYNAMIC TABLE IF EXISTS VOC_LAST_PURCHASE").collect()
# session.sql("DROP DYNAMIC TABLE IF EXISTS VOC_TICKET_EXTRACTIONS").collect()
# session.sql("DROP DYNAMIC TABLE IF EXISTS VOC_INTERACTION_TOPICS").collect()
# session.sql("DROP DYNAMIC TABLE IF EXISTS VOC_INTERACTION_SENTIMENT").collect()

# -- Base Tables --
# session.sql("DROP TABLE IF EXISTS VOC_SUPPORT_TICKETS").collect()
# session.sql("DROP TABLE IF EXISTS VOC_CHAT_INTERACTIONS").collect()
# session.sql("DROP TABLE IF EXISTS VOC_CALL_TRANSCRIPTS").collect()
# session.sql("DROP TABLE IF EXISTS VOC_PURCHASES").collect()
# session.sql("DROP TABLE IF EXISTS VOC_CUSTOMERS").collect()
# print("All VoC objects dropped")